# ARC NeuroGolf static ONNX solver 09- localish_recolor

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
required={'onnx':'onnx','onnxruntime':'onnxruntime','onnxsim':'onnxsim','torch':'torch','numpy':'numpy'}
missing=[pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install',*missing])


import json, os, random, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnx
import onnxruntime as ort
from onnxsim import simplify

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 102.1 MB/s eta 0:00:00


In [4]:
TASK_ID='task228'
CH=10
H=W=30

In [5]:

class CornerMoveStatic(nn.Module):
    def __init__(self):
        super().__init__()
        rr=torch.arange(H,dtype=torch.float32).view(1,1,H,1).expand(1,1,H,W)
        cc=torch.arange(W,dtype=torch.float32).view(1,1,1,W).expand(1,1,H,W)
        self.register_buffer('rr',rr); self.register_buffer('cc',cc)
    def forward(self,x):
        active=(x.sum(1,keepdim=True)>0).float()
        counts=x[:,1:].sum((2,3), keepdim=True)
        main_rel=counts.argmax(1,keepdim=True)+1
        main=sum([x[:,k:k+1]*(main_rel==k).float() for k in range(1,CH)])
        big=torch.full_like(self.rr,1000.0); small=torch.full_like(self.rr,-1000.0)
        r0=torch.where(main>0.5,self.rr,big).amin((2,3),keepdim=True)
        r1=torch.where(main>0.5,self.rr,small).amax((2,3),keepdim=True)
        c0=torch.where(main>0.5,self.cc,big).amin((2,3),keepdim=True)
        c1=torch.where(main>0.5,self.cc,small).amax((2,3),keepdim=True)
        inside=((self.rr>r0)&(self.rr<r1)&(self.cc>c0)&(self.cc<c1)).float()*active
        midr=(r0+r1)/2.0; midc=(c0+c1)/2.0
        top=(self.rr<=midr).float()*inside; bottom=(self.rr>midr).float()*inside
        left=(self.cc<=midc).float()*inside; right=(self.cc>midc).float()*inside
        tl=((self.rr==(r0-1))&(self.cc==(c0-1))).float()*active
        tr=((self.rr==(r0-1))&(self.cc==(c1+1))).float()*active
        bl=((self.rr==(r1+1))&(self.cc==(c0-1))).float()*active
        br=((self.rr==(r1+1))&(self.cc==(c1+1))).float()*active
        out=[torch.zeros_like(active) for _ in range(CH)]
        for k in range(1,CH):
            out[k]=main*(main_rel==k).float()
        for k in range(1,CH):
            m=x[:,k:k+1]*(1-(main_rel==k).float())*inside
            has_tl=((m*top*left).sum((2,3),keepdim=True)>0).float()
            has_tr=((m*top*right).sum((2,3),keepdim=True)>0).float()
            has_bl=((m*bottom*left).sum((2,3),keepdim=True)>0).float()
            has_br=((m*bottom*right).sum((2,3),keepdim=True)>0).float()
            add=br*has_tl + bl*has_tr + tr*has_bl + tl*has_br
            out[k]=(out[k]+add).clamp(0,1)
        occ=sum(out[1:]).clamp(0,1)
        out[0]=(1-occ)*active
        return torch.cat(out,1)


def load_task():
    for p in [Path.cwd()/f'{TASK_ID}.json', 
              Path.cwd().parent/f'{TASK_ID}.json', 
              Path('/mnt/data')/f'{TASK_ID}.json',
              Path(COMPETITION)/f'{TASK_ID}.json']:
        if p.exists():
            return json.load(open(p)), p
    raise FileNotFoundError(TASK_ID)

def onehot(grid):
    a=np.array(grid,dtype=np.int64); h,w=a.shape
    x=np.zeros((1,CH,H,W),dtype=np.float32)
    for k in range(CH): x[0,k,:h,:w]=(a==k)
    return x,h,w

def export_model(path):
    model=CornerMoveStatic().eval()
    torch.onnx.export(model, torch.zeros(1,CH,H,W), str(path), input_names=['input'], output_names=['output'], opset_version=13, dynamo=False)

def inspect_onnx(path):
    m=onnx.load(str(path)); ops={}
    for n in m.graph.node: ops[n.op_type]=ops.get(n.op_type,0)+1
    forbidden=[op for op in ['Loop','Scan','NonZero','Unique','Script','Function'] if ops.get(op)]
    risk=[op for op in ['ScatterND','Shape','Range','Expand','Gather','ConstantOfShape'] if ops.get(op)]
    return {'size':Path(path).stat().st_size,'ops':ops,'forbidden':forbidden,'risk':risk}

def validate(path,task):
    sess=ort.InferenceSession(str(path),providers=['CPUExecutionProvider'])
    rep={}
    for sec in ['train','test','arc-gen']:
        exact=0
        for ex in task[sec]:
            x,h,w=onehot(ex['input']); y=sess.run(None, {'input':x})[0].argmax(1)[0,:h,:w]
            exact += int(np.array_equal(y,np.array(ex['output'])))
        rep[sec]={'exact':exact,'total':len(task[sec])}
    return rep


In [6]:
task, task_path = load_task()
print('task path:', task_path)
print({k: len(v) for k,v in task.items() if isinstance(v,list)})

task path: /kaggle/input/competitions/neurogolf-2026/task228.json
{'train': 3, 'test': 1, 'arc-gen': 262}


In [7]:
OUT=Path.cwd()/'generated_models'
OUT.mkdir(exist_ok=True)
MODEL_PATH=OUT/f'{TASK_ID}.onnx'
export_model(MODEL_PATH)
arch=inspect_onnx(MODEL_PATH)
print(arch)
assert arch['size'] < 1_400_000
assert not arch['forbidden']
assert not arch['risk']

/tmp/ipykernel_16/2545615241.py:58: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, torch.zeros(1,CH,H,W), str(path), input_names=['input'], output_names=['output'], opset_version=13, dynamo=False)


{'size': 62661, 'ops': {'Constant': 130, 'ReduceSum': 38, 'Greater': 42, 'Cast': 55, 'Slice': 10, 'ArgMax': 1, 'Add': 57, 'Equal': 13, 'Mul': 136, 'Where': 4, 'ReduceMin': 2, 'ReduceMax': 2, 'Less': 2, 'And': 7, 'Div': 2, 'LessOrEqual': 2, 'Sub': 12, 'Clip': 10, 'Concat': 1}, 'forbidden': [], 'risk': []}


In [8]:
rep=validate(MODEL_PATH, task)
print(rep)
assert rep['train']['exact']==rep['train']['total']
assert rep['test']['exact']==rep['test']['total']
assert rep['arc-gen']['exact']==rep['arc-gen']['total']

{'train': {'exact': 3, 'total': 3}, 'test': {'exact': 1, 'total': 1}, 'arc-gen': {'exact': 262, 'total': 262}}


In [9]:
zip_path=Path.cwd()/'submission.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    zf.write(MODEL_PATH, arcname=f'{TASK_ID}.onnx')
print('submission:', zip_path)

submission: /kaggle/working/submission.zip
